In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path
import torch
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import DataLoader
from torch import nn
from torch.functional import F
from tqdm import tqdm
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from datasets import Dataset
import numpy as np

from src.data_utils import get_sample_from_row_original, filter_irrelevant, prune_frequent_samples
from src.inference_utils import predict
from src.metrics import lemmatization_accuracy

In [2]:
CURDIR = Path.cwd()

DATADIR = CURDIR / "data" / "original"
assert DATADIR.exists()

MODELS_DIR = CURDIR / "models"
assert MODELS_DIR.exists()

MODEL_ID = "ai-forever/ruT5-base"
# MODEL_ID = MODELS_DIR / "ruT5_base_tune"

RESULT_MODEL_DIR = MODELS_DIR / "ruT5_base_tune"
if not RESULT_MODEL_DIR.exists():
    RESULT_MODEL_DIR.mkdir()

MAX_LENGTH = 512
DEVICE = "cuda"

In [3]:
df_train = pd.read_csv(DATADIR / "train.csv", index_col=0, sep="\t")
df_dev = pd.read_csv(DATADIR / "dev.csv", index_col=0, sep="\t")

In [4]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_ID)

You are using the legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This means that tokens that come after special tokens will not be properly handled. We recommend you to read the related pull request available at https://github.com/huggingface/transformers/pull/24565


In [5]:
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID).to("cuda")

In [6]:
df_train["sample"] = df_train.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)
df_dev["sample"] = df_dev.apply(lambda row: get_sample_from_row_original(row)[0], axis=1)

df_train.shape, df_dev.shape

((2150060, 7), (255992, 7))

In [7]:
df_train.shape

(2150060, 7)

In [8]:
df_train = filter_irrelevant(df_train)
df_dev = filter_irrelevant(df_dev)

In [9]:
df_train.shape, df_dev.shape

((2135295, 7), (254996, 7))

In [10]:
df_train = prune_frequent_samples(df_train)  # здесь подрезаем несбалансированное начальное распределение
df_dev = df_dev.drop_duplicates(subset=["sample"])  # отсюда просто удаляем все дубли чтобы честно мериться

In [11]:
df_train.shape, df_dev.shape

((958668, 8), (52827, 7))

In [13]:
df_train["ln"] = df_train["sample"].map(len)
df_train = df_train[df_train["ln"] < 140]
df_train.shape, df_dev.shape

((958666, 9), (52827, 7))

In [14]:
df_train = df_train[["sample", "lemma"]]
df_dev = df_dev[["sample", "lemma"]]

In [15]:
train = Dataset.from_pandas(
    df_train[["sample", "lemma"]],
).rename_columns({
    "sample": "source",
    "lemma": "target",
}).shuffle(seed=42)

In [16]:
def tokenize_function(examples,):

    model_inputs = tokenizer(
        examples["source"],
        max_length=70,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        examples["target"],
        max_length=70,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [17]:
tokenized_train = train.map(
    tokenize_function,
    batched=True,
    batch_size=1000,
    remove_columns=train.column_names,
)

Map:   0%|          | 0/958666 [00:00<?, ? examples/s]

Map: 100%|██████████| 958666/958666 [01:51<00:00, 8561.12 examples/s]


In [18]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
    label_pad_token_id=-100,
)

In [19]:
BATCH_SIZE = 40  # 56? 48

In [20]:
train_dataloader = DataLoader(
    tokenized_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=1,
    pin_memory=True,
    drop_last=False,
)

In [21]:
def train_batch(
    model: T5ForConditionalGeneration,
    batch,
    optimizer,
):

    input_ids = batch['input_ids']
    attention_mask = batch['attention_mask']
    labels = batch['labels']

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels
    )
    loss = outputs.loss

    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    optimizer.zero_grad()

    return loss

In [22]:
def train_epoch(
    model: T5ForConditionalGeneration,
    iterator: DataLoader,
    optimizer,
    scheduler,
    device=DEVICE,
):

    model.eval()

    losses = []

    pbar = tqdm(iterator, desc="Training", unit="bs")

    for batch_id, batch in enumerate(pbar):
        batch = {k: val.to(device) for k, val in batch.items()}
        loss = train_batch(model, batch, optimizer)
        loss_item = loss.item()
        losses.append(loss_item)

        if batch_id % 100 == 0:
            pbar.set_description(f"Training (loss: {np.mean(losses):.6f})")
    scheduler.step()

    return np.mean(losses)

In [23]:
@torch.no_grad()
def validate(
    model: T5ForConditionalGeneration,
    tokenizer:T5Tokenizer,
    df: pd.DataFrame,
    device=DEVICE,
):
    model.eval()

    preds = predict(
        df["sample"].tolist(),
        model=model,
        tokenizer=tokenizer,
        device=device,
    )
    targets = df["lemma"].tolist()

    return lemmatization_accuracy(targets, preds)

In [24]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-2)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.65)

In [25]:
EPOCHS = 7

In [26]:
highest_val_acc = 0.5
patience = 0
train_losses = []
val_losses = []

max_patience = 2

for epoch in range(1, EPOCHS+1):

    train_loss = train_epoch(
        model,
        train_dataloader,
        optimizer,
        scheduler,
        DEVICE,
    )

    print(f"Train loss: {train_loss:.4f}")

    val_acc = validate(
        model,
        tokenizer,
        df_dev,
        DEVICE,
    )

    print(f"Val Acc: {val_acc:.4f}")

    if val_acc > highest_val_acc:
        model.save_pretrained(RESULT_MODEL_DIR)
    else:
        patience += 1
        print(f"patience {patience}/{max_patience}")
        if patience == max_patience:
            print("Early Stopping!")
            break

Training (loss: 0.099512): 100%|██████████| 23967/23967 [1:43:37<00:00,  3.86bs/s]


Train loss: 0.0994


1651it [03:14,  8.48it/s]                          


Val Acc: 0.9497


Training (loss: 0.031302): 100%|██████████| 23967/23967 [1:43:38<00:00,  3.85bs/s]


...

Val Acc: 0.9612


Training (loss: 0.010705): 100%|██████████| 23967/23967 [1:43:44<00:00,  3.85bs/s]


Train loss: 0.0107


 27%|██▋       | 443/1650 [00:52<02:23,  8.41it/s]


KeyboardInterrupt: 

In [27]:
val_acc = validate(
    model,
    tokenizer,
    df_dev,
    DEVICE,
)

print(f"Val Acc: {val_acc:.4f}")

1651it [03:12,  8.58it/s]                          

Val Acc: 0.9613


In [ ]:
# SMALL #
# Training (loss: 0.275460): 100%|██████████| 16529/16529 [35:57<00:00,  7.66bs/s]
# Train loss: 0.2753
# 1651it [01:22, 20.07it/s]                          
# Val Acc: 0.8510
# patience 1/2
# Training (loss: 0.153510): 100%|██████████| 16529/16529 [35:55<00:00,  7.67bs/s]
# Train loss: 0.1535
# 1651it [01:21, 20.14it/s]                          
# Val Acc: 0.8776
# patience 2/2
# Early Stopping!
# Training (loss: 0.125018): 100%|██████████| 16529/16529 [35:52<00:00,  7.68bs/s]
# Train loss: 0.1250
# 1651it [01:17, 21.23it/s]                          
# Val Acc: 0.8838
# patience 1/2
# Training (loss: 0.110583): 100%|██████████| 16529/16529 [35:52<00:00,  7.68bs/s]
# Train loss: 0.1106
# 1651it [01:18, 21.01it/s]                          
# Val Acc: 0.8914
# patience 2/2
# Early Stopping!

In [ ]:
# BASE #
# Training (loss: 0.099512): 100%|██████████| 23967/23967 [1:43:37<00:00,  3.86bs/s]
# Train loss: 0.0994
# 1651it [03:14,  8.48it/s]                          
# Val Acc: 0.9497
# Training (loss: 0.031302): 100%|██████████| 23967/23967 [1:43:38<00:00,  3.85bs/s]
# Train loss: 0.0313
# 1651it [03:14,  8.48it/s]                          
# Val Acc: 0.9580
# Training (loss: 0.020157): 100%|██████████| 23967/23967 [1:43:40<00:00,  3.85bs/s]
# Train loss: 0.0202
# 1651it [03:11,  8.62it/s]                          
# Val Acc: 0.9601
# Training (loss: 0.015073): 100%|██████████| 23967/23967 [1:43:41<00:00,  3.85bs/s]
# Train loss: 0.0151
# 1651it [03:12,  8.59it/s]                          
# Val Acc: 0.9607
# Training (loss: 0.012253): 100%|██████████| 23967/23967 [1:43:45<00:00,  3.85bs/s]
# Train loss: 0.0123
# 1651it [03:15,  8.45it/s]                          
# Val Acc: 0.9612
# Training (loss: 0.010705): 100%|██████████| 23967/23967 [1:43:44<00:00,  3.85bs/s]
# Train loss: 0.0107
#  27%|██▋       | 443/1650 [00:52<02:23,  8.41it/s]

In [ ]:
# model.save_pretrained(RESULT_MODEL_DIR)

In [ ]:
# tokenizer.save_pretrained(RESULT_MODEL_DIR)

('/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/ruT5_base_tune/tokenizer_config.json',
 '/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/ruT5_base_tune/special_tokens_map.json',
 '/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/ruT5_base_tune/spiece.model',
 '/home/smertlove/sandbox/hse/DEEPlom/FasterRubic2/models/ruT5_base_tune/added_tokens.json')

In [ ]:
# val_acc = validate(
#         teacher,
#         tokenizer,
#         df_dev,
#         DEVICE,
#     )

# print(f"Val Acc: {val_acc:.4f}")